# 02 — Model Evaluation

This notebook documents the machine-learning comparison used in the Sentinel-1 galamsey monitoring workflow.

### Models
- Logistic Regression
- Random Forest
- Support Vector Classifier (SVC)
- ExtraTrees

### Validation design
Training/tuning used reference plots **2, 3, 4, and 6**. Plots **1 and 5** were held out as spatially independent plots. `GroupKFold` was used during hyperparameter tuning, with recall as the RandomizedSearchCV scoring metric.

> **Important:** The current training script creates the held-out split but does not calculate held-out test predictions/metrics. Therefore this notebook does not invent such metrics.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

model_results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "SVC", "ExtraTrees"],
    "Accuracy": [0.84, 0.83, 0.82, 0.84],
    "Precision": [0.77, 0.76, 0.75, 0.77],
    "Recall": [0.96, 0.96, 0.96, 0.96],
    "F1": [0.86, 0.85, 0.85, 0.85],
}).set_index("Model")

display(model_results)

In [ ]:
ax = model_results.plot(kind="bar", figsize=(10, 5))
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Documented model performance")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Probability threshold analysis

The monitoring workflow converts model probabilities into binary detections using an operational threshold of **0.70**.

The documented performance at threshold 0.70 is:

In [ ]:
threshold_070 = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "SVC", "ExtraTrees"],
    "Accuracy": [0.851, 0.851, 0.847, 0.858],
    "Precision": [0.806, 0.799, 0.805, 0.816],
    "Recall": [0.924, 0.938, 0.916, 0.924],
    "F1": [0.861, 0.863, 0.857, 0.867],
}).set_index("Model")

display(threshold_070)

In [ ]:
best_thresholds = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "SVC", "ExtraTrees"],
    "Best threshold": [0.7, 0.8, 0.6, 0.7],
    "Accuracy": [0.851, 0.860, 0.849, 0.858],
    "Precision": [0.806, 0.816, 0.790, 0.816],
    "Recall": [0.924, 0.929, 0.951, 0.924],
    "F1": [0.861, 0.869, 0.863, 0.867],
}).set_index("Model")

display(best_thresholds)

## Feature importance

The documented feature-importance values were calculated at the project's stated importance level of 0.15:

In [ ]:
importance = pd.DataFrame({
    "VH": [1.80, 0.40, 0.43, 0.11],
    "VV": [1.74, 0.37, 0.39, 0.08],
    "NPI": [0.43, 0.08, 0.06, 0.00],
    "VV–VH difference": [0.30, 0.08, 0.06, 0.00],
    "VV/VH ratio": [0.19, 0.08, 0.06, 0.01],
}, index=["Logistic Regression", "Random Forest", "ExtraTrees", "SVC"])

display(importance)

## Spatial generalisation limitation

The thesis reports a major failure pattern on the spatially independent **Plot 5**, where galamsey recall dropped to zero across the four models.

This is an important qualification: strong results on the reference-sample evaluation do not by themselves establish robust spatial transfer to unseen locations.

This notebook therefore presents the documented results without fabricating held-out metrics that are not currently computed by the training script.